# Pandas Series Mini Project: E-Commerce Product Analysis

In this notebook, I wanted to move beyond very basic Series examples and build a small analysis that feels closer to a real project.

I created separate Pandas Series for product prices, units sold, returned products, stock levels, and customer ratings. Then I combined them by using their index labels.

My main goal here is to understand an important Pandas idea:

> Pandas does not only work by position. It aligns data by index labels.

This notebook focuses only on **Pandas Series**, but the logic will also be useful when I start working with DataFrames.


## What I Practice in This Notebook

- Creating Series with meaningful indexes
- Combining multiple Series
- Index alignment
- Revenue and net sales calculations
- Conditional filtering
- Sorting and ranking
- Missing-value handling
- Percentage calculations
- Detecting low-stock and high-return products
- Building a simple product performance score


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 1. Creating the Product Data

I use product names as index labels. This makes the analysis easier to read because I can access products directly by name instead of using only integer positions.


In [ ]:
products = [
    "Gaming Laptop",
    "Mechanical Keyboard",
    "Wireless Mouse",
    "27-inch Monitor",
    "USB-C Hub",
    "Noise-Cancelling Headphones"
]

prices = pd.Series(
    [54999, 3299, 1899, 12499, 1499, 8999],
    index=products,
    name="Price (TRY)"
)

units_sold = pd.Series(
    [34, 120, 175, 61, 98, 73],
    index=products,
    name="Units Sold"
)

returned_units = pd.Series(
    [2, 7, 15, 3, 11, 4],
    index=products,
    name="Returned Units"
)

stock = pd.Series(
    [9, 42, 18, 7, 65, 14],
    index=products,
    name="Current Stock"
)

ratings = pd.Series(
    [4.8, 4.5, 4.2, 4.7, np.nan, 4.6],
    index=products,
    name="Customer Rating"
)

prices

## 2. Checking the Series

Before calculating anything, I check the shape, indexes, data types, and missing values.

This is a small habit, but it helps me understand the data before I start the analysis.


In [ ]:
print("Number of products:", prices.size)
print("Price data type:", prices.dtype)
print("All indexes match:", prices.index.equals(units_sold.index))
print("Missing ratings:", ratings.isna().sum())

## 3. Gross Revenue

Gross revenue is calculated by multiplying each product's price by the number of units sold.

Because both Series use the same product names as indexes, Pandas matches the correct values automatically.


In [ ]:
gross_revenue = prices * units_sold
gross_revenue.name = "Gross Revenue (TRY)"

gross_revenue.sort_values(ascending=False)

## 4. Net Units Sold and Net Revenue

Returned products should not be counted as completed sales.

First, I subtract returned units from sold units. Then I calculate the net revenue.


In [ ]:
net_units_sold = units_sold - returned_units
net_units_sold.name = "Net Units Sold"

net_revenue = prices * net_units_sold
net_revenue.name = "Net Revenue (TRY)"

print("Net units sold:")
print(net_units_sold)

print("\nNet revenue:")
print(net_revenue.sort_values(ascending=False))

## 5. Return Rate

A high number of sales may look good at first, but a high return rate can show a product-quality or customer-expectation problem.

I calculate the return rate as a percentage of total units sold.


In [ ]:
return_rate = (returned_units / units_sold) * 100
return_rate.name = "Return Rate (%)"

return_rate.sort_values(ascending=False)

### Products with a Return Rate Above 8%

Here I use boolean filtering to find products that may need attention.


In [ ]:
high_return_products = return_rate[return_rate > 8]

high_return_products.sort_values(ascending=False)

## 6. Revenue Contribution

This calculation shows how much each product contributes to the total net revenue.

A product can have fewer sales but still make a large contribution because of its higher price.


In [ ]:
revenue_share = (net_revenue / net_revenue.sum()) * 100
revenue_share.name = "Revenue Share (%)"

revenue_share.sort_values(ascending=False)

## 7. Low-Stock Products

I want to find products that have strong sales but may run out of stock soon.

For this example, I define low stock as fewer than 15 units.


In [ ]:
low_stock = stock[stock < 15]

low_stock.sort_values()

### Urgent Restock Candidates

A product becomes an urgent restock candidate when:

- Its stock is below 15 units.
- Its net sales are above the average net sales.


In [ ]:
urgent_restock_mask = (stock < 15) & (net_units_sold > net_units_sold.mean())
urgent_restock_products = stock[urgent_restock_mask]

urgent_restock_products

## 8. Handling Missing Ratings

The USB-C Hub does not have a rating.

Instead of deleting the product, I fill the missing value with the average rating of the available products.


In [ ]:
average_rating = ratings.mean()
filled_ratings = ratings.fillna(average_rating)
filled_ratings.name = "Filled Customer Rating"

print("Average rating:", round(average_rating, 2))
print()
print(filled_ratings)

## 9. Index Alignment Example

To test index alignment, I intentionally create a discount Series in a different order.

Pandas still matches the discount rate with the correct product by using index labels.


In [ ]:
discount_rates = pd.Series(
    {
        "USB-C Hub": 0.10,
        "Gaming Laptop": 0.05,
        "Noise-Cancelling Headphones": 0.15,
        "Wireless Mouse": 0.08,
        "27-inch Monitor": 0.12,
        "Mechanical Keyboard": 0.07
    },
    name="Discount Rate"
)

discounted_prices = prices * (1 - discount_rates)
discounted_prices.name = "Discounted Price (TRY)"

discounted_prices.sort_values(ascending=False)

## 10. Estimated Revenue After Discounts

This is a simplified scenario where I assume the same net sales volume after applying discounts.


In [ ]:
discounted_net_revenue = discounted_prices * net_units_sold
discounted_net_revenue.name = "Discounted Net Revenue (TRY)"

revenue_difference = net_revenue - discounted_net_revenue
revenue_difference.name = "Revenue Difference (TRY)"

print("Estimated discounted revenue:")
print(discounted_net_revenue.sort_values(ascending=False))

print("\nEstimated revenue difference:")
print(revenue_difference.sort_values(ascending=False))

## 11. Ranking Products

I rank products by net revenue.

The product with rank 1 generates the highest net revenue.


In [ ]:
revenue_rank = net_revenue.rank(method="dense", ascending=False).astype(int)
revenue_rank.name = "Revenue Rank"

revenue_rank.sort_values()

## 12. Creating a Simple Performance Score

I create a small scoring system by combining:

- Revenue share
- Customer rating
- Return-rate quality

Before combining them, I normalize each metric to a similar scale.

This is not a real business model. It is only a practice example to understand how multiple Series can be combined.


In [ ]:
normalized_revenue = net_revenue / net_revenue.max()
normalized_rating = filled_ratings / 5
return_quality = 1 - (return_rate / 100)

performance_score = (
    normalized_revenue * 0.50
    + normalized_rating * 0.30
    + return_quality * 0.20
) * 100

performance_score.name = "Performance Score"

performance_score.sort_values(ascending=False)

## 13. Best and Weakest Performing Products

The `idxmax()` and `idxmin()` methods return the index label of the highest and lowest values.


In [ ]:
best_product = performance_score.idxmax()
weakest_product = performance_score.idxmin()

print("Best-performing product:", best_product)
print("Score:", round(performance_score[best_product], 2))

print("\nWeakest-performing product:", weakest_product)
print("Score:", round(performance_score[weakest_product], 2))

## 14. Final Business Summary

Here I create a short summary by using the results of the analysis.


In [ ]:
print("E-COMMERCE PRODUCT ANALYSIS")
print("-" * 35)
print(f"Total gross revenue: {gross_revenue.sum():,.2f} TRY")
print(f"Total net revenue: {net_revenue.sum():,.2f} TRY")
print(f"Total returned units: {returned_units.sum()}")
print(f"Average return rate: {return_rate.mean():.2f}%")
print(f"Highest-revenue product: {net_revenue.idxmax()}")
print(f"Most returned product: {return_rate.idxmax()}")
print(f"Best performance score: {performance_score.idxmax()}")
print(f"Products requiring urgent restock: {urgent_restock_products.index.tolist()}")

## 15. Mini Challenge

I leave a few questions for myself so I can return to this notebook and solve them without looking at the previous cells.

1. Find products with a rating above 4.5 and a return rate below 6%.
2. Apply a 12% price increase only to products with stock below 15.
3. Find the product whose net revenue is closest to the average net revenue.
4. Calculate how many units each product should restock to reach 50 units.
5. Create a new score using 40% revenue, 40% rating, and 20% stock availability.


## My Notes

- Series indexes are not just labels; Pandas uses them while combining data.
- Boolean masks make it possible to apply more than one condition at the same time.
- Methods such as `idxmax()`, `idxmin()`, `rank()`, `sort_values()`, and `fillna()` are very useful in analysis.
- High sales alone do not mean that a product is successful. Return rate, stock, rating, and revenue should also be considered.
- Normalization is useful when I want to combine metrics that use different scales.
- This project helped me see how separate Series can work together like connected columns.


## Key Takeaways

- A Pandas Series is a one-dimensional data structure with labeled indexes.
- Index alignment is one of the most important Pandas concepts.
- Series can be combined to produce new calculated metrics.
- Filtering, sorting, ranking, and missing-value handling are essential analysis operations.
- Even without using a DataFrame, I can build a meaningful small analysis with multiple Series.
- The next logical step is to move the same project into a Pandas DataFrame and compare the workflow.
